# 1. Tratamento de dados

Os passos a seguir tem por objetivo tornar os dados adequados às etapas seguintes.

In [19]:
# importa bibliotecas necessárias para tratamento de dados
import pandas as pd

In [20]:
# instancia os dados em um dataframe
df = pd.read_csv('data/planilha_usinagem_dureza.csv', sep=';')

In [21]:
# exibe informações sobre as colunas
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 13 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Data                                       242 non-null    str    
 1   Fabricante do Hob                          244 non-null    str    
 2   Revestimento                               242 non-null    str    
 3   Fabricante                                 242 non-null    str    
 4   Avanço mm/rev.                             244 non-null    str    
 5   Rotação RPM                                244 non-null    int64  
 6   Vc m/min                                   244 non-null    float64
 7   Shifting mm                                243 non-null    float64
 8   Sub-Shift mm                               243 non-null    float64
 9   CT (S)                                     244 non-null    str    
 10  Quantidade de pçs por afiação Target 

Este material tem por objetivo montar uma lógica de apredizado de máquina e usa uma base dedos preenchida manualmente e obtida, e não criada, por este desenvolvedor. Por este motivo, optou-se pelo descarte dos dados faltantes. Em trabalhos futuros objetiva-se criar mecanismos de obtenção de dados mais eficazes a fim de garantir a maior robustez dos dados.

In [22]:
# exclue todas as linhas com itens faltantes
df_limpo = df.dropna()

In [23]:
# exibe informações sobre as colunas
df_limpo.info()

<class 'pandas.DataFrame'>
Index: 105 entries, 3 to 234
Data columns (total 13 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Data                                       105 non-null    str    
 1   Fabricante do Hob                          105 non-null    str    
 2   Revestimento                               105 non-null    str    
 3   Fabricante                                 105 non-null    str    
 4   Avanço mm/rev.                             105 non-null    str    
 5   Rotação RPM                                105 non-null    int64  
 6   Vc m/min                                   105 non-null    float64
 7   Shifting mm                                105 non-null    float64
 8   Sub-Shift mm                               105 non-null    float64
 9   CT (S)                                     105 non-null    str    
 10  Quantidade de pçs por afiação Target 1950 

In [24]:
# exibe os valores únicos de cada coluna

for col in df_limpo.columns: # loop para iterar sobre todas as colunas
    print(f"Valores únicos de '{col}'") # imprime o nome da coluna
    print(df_limpo[col].unique()) # imprime os valores únicos da coluna
    print('----------------------------------------------------------------------') # imprime um separador entre cada iteração

Valores únicos de 'Data'
<StringArray>
['2025-11-17 00:00:00', '2025-11-18 00:00:00', '2025-11-19 00:00:00',
 '2025-11-26 00:00:00', '2025-11-29 00:00:00', '2025-12-01 00:00:00',
 '2025-12-02 00:00:00', '2025-12-04 00:00:00', '2025-12-05 00:00:00',
 '2025-12-06 00:00:00', '2025-12-11 00:00:00', '2025-12-12 00:00:00',
 '2025-12-16 00:00:00', '2025-12-18 00:00:00', '2025-12-19 00:00:00',
 '2025-12-20 00:00:00', '2025-12-22 00:00:00', '2025-12-24 00:00:00',
 '2026-01-03 00:00:00', '2026-01-06 00:00:00', '2026-01-07 00:00:00',
 '2026-01-09 00:00:00', '2026-01-10 00:00:00', '2026-01-12 00:00:00',
 '2026-01-13 00:00:00', '2026-01-18 00:00:00', '2026-01-19 00:00:00',
 '2026-01-21 00:00:00', '2026-01-23 00:00:00', '2026-01-24 00:00:00',
 '2026-01-25 00:00:00', '2026-01-27 00:00:00', '2026-01-28 00:00:00',
 '2026-01-30 00:00:00', '2026-02-02 00:00:00', '2026-02-03 00:00:00',
 '2026-02-04 00:00:00', '2026-02-05 00:00:00', '2026-02-06 00:00:00',
 '2026-02-07 00:00:00', '2026-02-08 00:00:00', '202

In [25]:
# remove sujeira da coluna 'Avanço mm/rev.'

df_limpo = df_limpo[df_limpo['Avanço mm/rev.'] != '\\'] # remove a linha com '\' da colunas 'Avanço mm/rev.'
print("Valores únicos da coluna 'Avanço mm/rev.'") # uma pequena descrição
print(df_limpo['Avanço mm/rev.'].unique()) # imprime os valores únicos da coluna

Valores únicos da coluna 'Avanço mm/rev.'
<StringArray>
['2', '1.4', '1.5', '1.75']
Length: 4, dtype: str


In [26]:
df_limpo['Fabricante do Hob'].value_counts()

Fabricante do Hob
SU       102
S.U.       1
Nidec      1
Name: count, dtype: int64

Das 104 entradas remanecentes em df_limpo, apenas 1 refere-se a 'Fabricante do Hob' Nidec. Por este motivo, será considerada como coluna de valor único. Além disso, para este estudo, também será descartada a coluna data visto que o modelo não usará séries temporais

In [27]:
# elimina as colunas de valor único
df_limpo = df_limpo.drop(columns=['Data', 'Fabricante', 'Fabricante do Hob'])

In [28]:
# descarta possíveis linhas duplicadasd
df_limpo = df_limpo.drop_duplicates()

In [29]:
# converte as colunas faltantes para os tipos finais

colunas_converter_float = ['Avanço mm/rev.', 'CT (S)'] # define as colunas a serem corrigidas

for coluna in colunas_converter_float: # loop para iterar sobre as colunas a sere corrigidas
    df_corrigido = df_limpo # copia df_limpo para um novo dataframe
    df_corrigido[coluna] = df_corrigido[coluna].astype(float) # converte as colunas em colunas_converter_float para float

df_corrigido['Quantidade de pçs por afiação Target 1950'] = df_corrigido['Quantidade de pçs por afiação Target 1950'].astype(int) # converte a 'Quantidade de pçs por afiação Target 1950' para int64

df_corrigido.info() # exibe informações sobre as colunas

<class 'pandas.DataFrame'>
Index: 104 entries, 3 to 234
Data columns (total 10 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Revestimento                               104 non-null    str    
 1   Avanço mm/rev.                             104 non-null    float64
 2   Rotação RPM                                104 non-null    int64  
 3   Vc m/min                                   104 non-null    float64
 4   Shifting mm                                104 non-null    float64
 5   Sub-Shift mm                               104 non-null    float64
 6   CT (S)                                     104 non-null    float64
 7   Quantidade de pçs por afiação Target 1950  104 non-null    int64  
 8   dureza sup                                 104 non-null    float64
 9   dureza nuc                                 104 non-null    float64
dtypes: float64(7), int64(2), str(1)
memory usa

In [30]:
df_corrigido

,Revestimento,Avanço mm/rev.,Rotação RPM,Vc m/min,Shifting mm,Sub-Shift mm,CT (S),Quantidade de pçs por afiação Target 1950,dureza sup,dureza nuc
3,Alcrona Pro,2.0,510,144.126000,10.0,0.666667,25.733333,1644,97.50,95.50
4,Alcrona Evo,2.0,510,144.126000,11.4,0.876923,25.733333,1500,94.50,94.00
5,Alcrona Pro,2.0,510,144.126000,11.4,0.876923,25.733333,1440,96.50,95.50
10,Alcrona Evo,2.0,510,144.126000,11.4,0.876923,25.733333,1521,98.25,97.50
13,Alcrona Evo,2.0,510,144.126000,11.4,0.876923,25.733333,1341,96.00,94.65
...,...,...,...,...,...,...,...,...,...,...
219,Alcrona Pro,2.0,510,144.199103,11.4,0.876923,25.800000,1047,93.75,89.95
220,Alcrona Pro,2.0,510,144.199103,11.4,0.876923,25.800000,1632,93.50,87.80
221,Alcrona Pro,2.0,510,144.199103,11.4,0.876923,25.800000,1209,94.40,86.50
230,Alcrona Pro,2.0,510,144.199103,11.4,0.876923,25.800000,1470,92.55,89.70


# 2. Análise Exploratória dos Dados (EDA)

In [31]:
# importa bibliotecas necessárias para a EDA
import plotly.express as px

In [32]:
# cria um histograma da coluna 'Quantidade de pçs por afiação Target 1950'

fig = px.histogram( # cria uma figura para o gráfico
    df_corrigido, # define o df a ser usado
    x='Quantidade de pçs por afiação Target 1950', # define os valores de X
    nbins=10, # sugestioná o número de barras do hostograma usando o método da raiz quadrada (k = n^(1/2) ==> nbins = 104 linhas^(1/2))
    title='Distribuição da Quantidade de Peças por Afiação', # define um título para o gráfico
    labels={'Quantidade de pçs por afiação Target 1950': 'Quantidade de Peças'}, # múda o título do eixo X de 'Quantidade de pçs por afiação Target 1950' para 'Quantidade de Peças'
    color_discrete_sequence=['#1f77b4'] # define a cor padrão como azul
)

fig.update_layout(yaxis_title='Frequência') # muda o título do eixo Y de 'count' para 'Frequência'

fig.show() # exibe a figura

Através do histograma da distribuição da quantidade de peças por afiação é possível observar um comportamento normal em quase todas as faixas com excessão da faixa de 3500 a 3999 peças por afiação que apresenta uma única insidência. O que pode indicar um erro de durante a aquisição desse dado (erro de digitação, por exemplo) ou um evento anormalmente favorável à durabilidade da ferramenta.

In [33]:
# cria um gráfico de boxplot da relação entre 'Revestimento' e 'Quantidade de pçs por afiação Target 1950'

fig_box = px.box( # cria uma figura para o boxplot
    df_corrigido, # define o df a ser usado
    x='Revestimento', # define os valores de X
    y='Quantidade de pçs por afiação Target 1950', # define os valores de Y 
    color='Revestimento', # separa os dados conforme 'Revestimento'
    title='Variação da Vida Útil por Revestimento', # define um título para o gráfico
    labels={'Quantidade de pçs por afiação Target 1950': 'Peças Produzidas'} #  # múda o título do eixo X de 'Quantidade de pçs por afiação Target 1950' para 'Quantidade de Peças'
)

fig_box.show() # exibe a figura

Apesar de ser mais notável no Revestimento 'Alcrona Evo', ambos os revestimentosapresentam apresentam uma variação muito maior na vida útil do que o esperado. É necessessário investigar o motivo desta variação.

In [34]:
# cria um heatmap das variáveis numéricas de df_corrigido

df_numerico = df_corrigido.select_dtypes(include=['float64', 'int64']) # cria df_numerico com apneas as colunas numéricas de df_corrigido

matriz_correlacao = df_numerico.corr(method='spearman') # cria uma matriz de correlação das colunas de df_numerico

fig_corr = px.imshow( # cria uma figura para o heatmap
    matriz_correlacao, # define a matriz a ser usado
    text_auto=".2f", # mostra os valores com 2 casas decimais
    aspect="auto", # define a proporção do heatmap como automática forçañdo-o a preencher a altura e largura da figura
    color_continuous_scale='RdBu_r', # define a escala de cores como Azul para positivo e Vermelho para negativo
    title='Matriz de Correlação entre Parâmetros de Usinagem' # define o título do gráfico
)

fig_corr.show() # exibe a figura

Não foi possível identificar nenhuma correlação linear direta entre as features e a variável alvo 'Quantidade de pçs por afiação Target 1950'. No entanto algumas features apresentaram relações fortíssimas. Relações desse tipo indicam redundancia nas medições e pode ser descartadas para melhorar a eficiência do futuro modelo preditivo. Por este motivo serão descartadas as seguintes colunas:

* 'Vc m/min' : Correlação de 99% com 'Rotação RPM'
* 'CT (s)' : Correlação de 99% com 'Rotação RPM'
* 'Sub-shift mm' :  Correlação de 100% com 'Shifting mm'

In [35]:
df_corrigido

,Revestimento,Avanço mm/rev.,Rotação RPM,Vc m/min,Shifting mm,Sub-Shift mm,CT (S),Quantidade de pçs por afiação Target 1950,dureza sup,dureza nuc
3,Alcrona Pro,2.0,510,144.126000,10.0,0.666667,25.733333,1644,97.50,95.50
4,Alcrona Evo,2.0,510,144.126000,11.4,0.876923,25.733333,1500,94.50,94.00
5,Alcrona Pro,2.0,510,144.126000,11.4,0.876923,25.733333,1440,96.50,95.50
10,Alcrona Evo,2.0,510,144.126000,11.4,0.876923,25.733333,1521,98.25,97.50
13,Alcrona Evo,2.0,510,144.126000,11.4,0.876923,25.733333,1341,96.00,94.65
...,...,...,...,...,...,...,...,...,...,...
219,Alcrona Pro,2.0,510,144.199103,11.4,0.876923,25.800000,1047,93.75,89.95
220,Alcrona Pro,2.0,510,144.199103,11.4,0.876923,25.800000,1632,93.50,87.80
221,Alcrona Pro,2.0,510,144.199103,11.4,0.876923,25.800000,1209,94.40,86.50
230,Alcrona Pro,2.0,510,144.199103,11.4,0.876923,25.800000,1470,92.55,89.70


In [36]:
# seleciona as features que serão usadas no modelo

df_selecionado = df_corrigido # copia o df_corrigido em df_selecionado
df_selecionado = df_selecionado.drop(columns=['Vc m/min', 'CT (S)', 'Sub-Shift mm']) # descarta as colunas 'Vc m/min', 'CT (S)' e 'Sub-Shift mm'
df_selecionado.head() # exibe df_selecionado

,Revestimento,Avanço mm/rev.,Rotação RPM,Shifting mm,Quantidade de pçs por afiação Target 1950,dureza sup,dureza nuc
3,Alcrona Pro,2.0,510,10.0,1644,97.50,95.50
4,Alcrona Evo,2.0,510,11.4,1500,94.50,94.00
5,Alcrona Pro,2.0,510,11.4,1440,96.50,95.50
10,Alcrona Evo,2.0,510,11.4,1521,98.25,97.50
13,Alcrona Evo,2.0,510,11.4,1341,96.00,94.65


# 3. Feature Engineering (Encoding e divisão target-features)

In [44]:
# One Hot Encoding na variável categórica 'Revestimento'
df_modelo = pd.get_dummies(df_selecionado, columns=['Revestimento'], drop_first=True, dtype=int).reset_index(drop=True)

In [45]:
# divisão target-features de df_modelo

X = df_modelo.drop(columns=['Quantidade de pçs por afiação Target 1950']) # separa todas as features em X
y = df_modelo['Quantidade de pçs por afiação Target 1950'] # separa o alvo em y